# Sanity Check: AI Business Operations Assistant for Online Transaction Risk

**Goal:** test whether the proposed *fraud learning + business operations assistant* is technically plausible with the mounted Kaggle datasets:

- `/kaggle/input/datasets/ealaxi/paysim1` — PaySim
- IEEE-CIS Fraud Detection — discovered automatically under `/kaggle/input`

The notebook checks dataset structure, fraud labels, temporal behavior, behavioral features, leakage, a lightweight model, and a cost-aware `APPROVE / REVIEW / BLOCK` decision layer.

IEEE-CIS contains transaction and identity tables joined by `TransactionID`; not all transactions have identity information, and Kaggle reports 871 columns overall. `TransactionDT` is a relative timedelta. citeturn0search0turn0search1

PaySim contains transaction type, amount, origin/destination accounts, time steps, and fraud labels. Kaggle specifically warns that the four balance columns should not be used for fraud detection because fraudulent transactions are cancelled in the simulation. citeturn0search2

In [ ]:
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_SAMPLE = 100_000

PAYSIM_ROOT = Path("/kaggle/input/datasets/ealaxi/paysim1")

print("PaySim path exists:", PAYSIM_ROOT.exists())
print("Kaggle input exists:", Path("/kaggle/input").exists())


## 1. Discover the mounted datasets

The IEEE-CIS competition provides transaction and identity CSVs, with the two tables joined through `TransactionID`. citeturn0search0

In [ ]:
all_csv = [Path(p) for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)]

print(f"CSV files found: {len(all_csv)}")
for p in all_csv[:100]:
    print(p)

def find_file(filename):
    hits = [p for p in all_csv if p.name.lower() == filename.lower()]
    return hits[0] if hits else None

IEEE_TRAIN_TX = find_file("train_transaction.csv")
IEEE_TRAIN_ID = find_file("train_identity.csv")
IEEE_TEST_TX = find_file("test_transaction.csv")
IEEE_TEST_ID = find_file("test_identity.csv")

print("\nDetected IEEE-CIS:")
for name, path in {
    "train_transaction": IEEE_TRAIN_TX,
    "train_identity": IEEE_TRAIN_ID,
    "test_transaction": IEEE_TEST_TX,
    "test_identity": IEEE_TEST_ID,
}.items():
    print(f"{name:20s}: {path}")

paysim_csvs = list(PAYSIM_ROOT.rglob("*.csv")) if PAYSIM_ROOT.exists() else []
print("\nPaySim CSV files:")
for p in paysim_csvs:
    print(p)

PAYSIM_FILE = paysim_csvs[0] if paysim_csvs else None


## 2. Load lightweight samples

IEEE-CIS is large, so this first pass reads only 100,000 rows. Full-data processing should be done only after the sanity check succeeds.

In [ ]:
def read_sample(path, nrows=N_SAMPLE):
    if path is None:
        return None
    return pd.read_csv(path, nrows=nrows, low_memory=False)

ieee_tx = read_sample(IEEE_TRAIN_TX)
ieee_id = read_sample(IEEE_TRAIN_ID)
paysim = read_sample(PAYSIM_FILE)

print("IEEE transaction:", None if ieee_tx is None else ieee_tx.shape)
print("IEEE identity   :", None if ieee_id is None else ieee_id.shape)
print("PaySim          :", None if paysim is None else paysim.shape)


## 3. Schema inspection

IEEE-CIS documents fields such as `ProductCD`, `card1-card6`, `addr1/addr2`, email domains, `M1-M9`, `DeviceType`, `DeviceInfo`, and `id_12-id_38`. Its `C*`, `D*`, `M*`, and `V*` variables contain masked/engineered behavioral and relationship information. citeturn0search5

PaySim documents `step`, `type`, `amount`, `nameOrig`, `oldbalanceOrg`, `newbalanceOrig`, `nameDest`, `oldbalanceDest`, `newbalanceDest`, `isFraud`, and `isFlaggedFraud`. citeturn0search2

In [ ]:
def schema_report(df, name):
    print(f"\n===== {name} =====")
    if df is None:
        print("NOT FOUND")
        return
    print("Shape:", df.shape)
    print("\nDtype counts:")
    print(df.dtypes.value_counts())
    print("\nFirst 100 columns:")
    print(df.columns.tolist()[:100])
    print("\nDuplicate rows:", df.duplicated().sum())

schema_report(ieee_tx, "IEEE-CIS train_transaction sample")
schema_report(ieee_id, "IEEE-CIS train_identity sample")
schema_report(paysim, "PaySim sample")


## 4. Fraud target and class imbalance

In [ ]:
def target_report(df, target="isFraud", name="dataset"):
    if df is None or target not in df.columns:
        print(f"{name}: target not found")
        return None
    vc = df[target].value_counts(dropna=False)
    pct = df[target].value_counts(normalize=True, dropna=False) * 100
    out = pd.DataFrame({"count": vc, "percent": pct.round(4)})
    print(f"\n{name}")
    display(out)
    if len(vc) >= 2:
        print("Majority/minority ratio:", round(vc.max() / vc.min(), 2))
    return out

ieee_target = target_report(ieee_tx, name="IEEE-CIS")
paysim_target = target_report(paysim, name="PaySim")


## 5. Missingness and cardinality

In [ ]:
def profile_columns(df):
    if df is None:
        return None
    prof = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })
    prof["unique_pct"] = (prof["n_unique"] / max(len(df), 1) * 100).round(2)
    prof["constant"] = prof["n_unique"] <= 1
    return prof.sort_values(["missing_pct", "n_unique"], ascending=[False, False])

ieee_profile = profile_columns(ieee_tx)
paysim_profile = profile_columns(paysim)

print("IEEE-CIS highest missingness:")
display(ieee_profile.head(30) if ieee_profile is not None else pd.DataFrame())

print("\nPaySim profile:")
display(paysim_profile if paysim_profile is not None else pd.DataFrame())


## 6. Temporal sanity check

IEEE-CIS uses `TransactionDT` as a relative timedelta. PaySim `step` represents one hour and spans 744 steps / 30 days. citeturn0search1turn0search2

In [ ]:
if ieee_tx is not None and "TransactionDT" in ieee_tx.columns:
    print("IEEE TransactionDT summary:")
    display(ieee_tx["TransactionDT"].describe())

    if "isFraud" in ieee_tx.columns:
        tmp = ieee_tx[["TransactionDT", "isFraud"]].copy()
        q = min(20, max(2, tmp["TransactionDT"].nunique()))
        tmp["time_bin"] = pd.qcut(tmp["TransactionDT"], q=q, duplicates="drop")
        temporal = tmp.groupby("time_bin", observed=True)["isFraud"].agg(["count", "mean"])
        display(temporal)

        temporal["mean"].plot(figsize=(10,4), marker="o")
        plt.title("IEEE-CIS sampled fraud rate over relative time")
        plt.xlabel("Time bin")
        plt.ylabel("Fraud rate")
        plt.grid(alpha=0.2)
        plt.show()

if paysim is not None and "step" in paysim.columns:
    print("PaySim step range:", paysim["step"].min(), "to", paysim["step"].max())
    if "isFraud" in paysim.columns and "type" in paysim.columns:
        display(paysim.groupby("type")["isFraud"].agg(["count","mean"]).sort_values("mean", ascending=False))


## 7. Behavioral features for PaySim

This directly tests the proposed **fraud-learning operations assistant**.

We build historical, pre-transaction signals:

- previous origin-account transaction count
- previous origin-account average amount
- previous origin-account amount standard deviation
- amount relative to previous origin behavior
- previous destination-account transaction count

`shift(1)` is used so the current transaction does not contaminate its own history.

Kaggle explicitly says the four balance fields should not be used for PaySim fraud detection. citeturn0search2

In [ ]:
def build_paysim_behavior_features(df):
    if df is None:
        return None

    d = df.copy()

    if "step" in d.columns:
        d = d.sort_values(["nameOrig", "step"], kind="stable").reset_index(drop=True)

    if "nameOrig" in d.columns:
        d["orig_prev_tx_count"] = d.groupby("nameOrig").cumcount()
        d["orig_prev_amount_mean"] = (
            d.groupby("nameOrig")["amount"]
             .transform(lambda s: s.shift(1).expanding().mean())
        )
        d["orig_prev_amount_std"] = (
            d.groupby("nameOrig")["amount"]
             .transform(lambda s: s.shift(1).expanding().std())
        )

    if "nameDest" in d.columns:
        d["dest_prev_tx_count"] = d.groupby("nameDest").cumcount()

    if "orig_prev_amount_mean" in d.columns:
        d["amount_vs_orig_mean"] = d["amount"] / (d["orig_prev_amount_mean"] + 1e-6)

    return d

paysim_b = build_paysim_behavior_features(paysim)

if paysim_b is not None:
    show_cols = [c for c in [
        "step","type","amount","nameOrig","nameDest",
        "orig_prev_tx_count","orig_prev_amount_mean",
        "orig_prev_amount_std","amount_vs_orig_mean",
        "dest_prev_tx_count","isFraud"
    ] if c in paysim_b.columns]
    display(paysim_b[show_cols].head(15))


## 8. Leakage audit

**PaySim exclusions:** `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`.

These are excluded because the dataset documentation warns that fraudulent transactions are cancelled. citeturn0search2

For both datasets, behavioral features must only use information available before the current transaction. For IEEE-CIS, be especially careful with future aggregation and target encoding.

In [ ]:
PAYSIM_FORBIDDEN = {
    "oldbalanceOrg","newbalanceOrig",
    "oldbalanceDest","newbalanceDest"
}

if paysim is not None:
    print("PaySim forbidden columns present:",
          sorted(PAYSIM_FORBIDDEN.intersection(paysim.columns)))

if ieee_tx is not None:
    print("IEEE target present:", "isFraud" in ieee_tx.columns)
    print("IEEE TransactionID:", "TransactionID" in ieee_tx.columns)
    print("IEEE TransactionDT:", "TransactionDT" in ieee_tx.columns)


## 9. Lightweight PaySim baseline

This is only a feasibility test.

Question:

> Do simple transaction + historical-behavior features contain enough signal to support the business-action layer?

A time-aware split is used.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

if paysim_b is not None and "isFraud" in paysim_b.columns:
    candidate_features = [
        "step","amount",
        "orig_prev_tx_count",
        "orig_prev_amount_mean",
        "orig_prev_amount_std",
        "amount_vs_orig_mean",
        "dest_prev_tx_count"
    ]
    feature_cols = [c for c in candidate_features if c in paysim_b.columns]

    model_df = paysim_b[feature_cols + ["isFraud"]].replace(
        [np.inf, -np.inf], np.nan
    ).copy()

    cutoff = model_df["step"].quantile(0.80)
    train_mask = model_df["step"] <= cutoff
    test_mask = model_df["step"] > cutoff

    X_train = model_df.loc[train_mask, feature_cols]
    y_train = model_df.loc[train_mask, "isFraud"]
    X_test = model_df.loc[test_mask, feature_cols]
    y_test = model_df.loc[test_mask, "isFraud"]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=150,
            learning_rate=0.08,
            max_leaf_nodes=31,
            random_state=RANDOM_STATE
        ))
    ])

    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:,1]
    pred = (prob >= 0.5).astype(int)

    print("Features:", feature_cols)
    print("Train:", X_train.shape, "Test:", X_test.shape)
    print("Train fraud rate:", round(y_train.mean(), 6))
    print("Test fraud rate :", round(y_test.mean(), 6))
    print("ROC-AUC:", round(roc_auc_score(y_test, prob), 5))
    print("PR-AUC :", round(average_precision_score(y_test, prob), 5))
    print(classification_report(y_test, pred, digits=4))
else:
    print("Baseline skipped: PaySim or isFraud not found.")


## 10. Business decision layer

Instead of only `fraud / legitimate`, the assistant produces:

- **APPROVE** — low risk
- **REVIEW** — uncertain/moderate risk
- **BLOCK** — high risk

The thresholds are illustrative and should be optimized in the final study.

In [ ]:
def business_actions(probability, review_threshold=0.30, block_threshold=0.75):
    p = np.asarray(probability)
    return np.where(
        p < review_threshold,
        "APPROVE",
        np.where(p < block_threshold, "REVIEW", "BLOCK")
    )

if "prob" in globals() and "y_test" in globals():
    actions = business_actions(prob)

    print("Recommended action counts:")
    display(pd.Series(actions).value_counts())

    decision_df = pd.DataFrame({
        "fraud_probability": prob,
        "actual_fraud": y_test.to_numpy(),
        "recommended_action": actions
    })

    display(decision_df.head(20))

    print("Fraud rate by recommended action:")
    display(
        decision_df.groupby("recommended_action")["actual_fraud"]
        .agg(["count","mean"])
        .sort_values("mean", ascending=False)
    )


## 11. Cost-aware decision sanity check

A business has different costs for:

- approving fraud
- blocking legitimate customers
- sending a transaction to manual review

Therefore, threshold selection can optimize **business cost**, not just accuracy.

In [ ]:
def evaluate_business_cost(
    y_true, prob, review_t, block_t,
    fn_cost=20.0, fp_cost=2.0, review_cost=0.5
):
    y_true = np.asarray(y_true)
    actions = business_actions(prob, review_t, block_t)
    total = 0.0

    for y, a in zip(y_true, actions):
        if a == "APPROVE" and y == 1:
            total += fn_cost
        elif a == "BLOCK" and y == 0:
            total += fp_cost
        elif a == "REVIEW":
            total += review_cost

    return total / len(y_true)

if "prob" in globals() and "y_test" in globals():
    rows = []
    for review_t in np.arange(0.10, 0.71, 0.05):
        for block_t in np.arange(max(review_t + 0.05, 0.30), 0.96, 0.05):
            rows.append([
                review_t, block_t,
                evaluate_business_cost(
                    y_test, prob, review_t, block_t,
                    fn_cost=20.0, fp_cost=2.0, review_cost=0.5
                )
            ])

    cost_df = pd.DataFrame(
        rows,
        columns=["review_threshold","block_threshold","avg_business_cost"]
    )

    display(cost_df.sort_values("avg_business_cost").head(15))


## 12. Explainability readiness

For the final paper, SHAP can convert a high-risk score into operational reasons such as:

- unusual transaction amount
- abnormal transaction velocity
- unusual historical behavior
- unusual destination activity

This is what makes the system an **operations assistant**, rather than another black-box fraud classifier.

In [ ]:
if "feature_cols" in globals():
    print("Candidate explanation features:")
    for c in feature_cols:
        print(" -", c)

print("Example operational output:")
print("Risk: HIGH")
print("Action: REVIEW")
print("Reasons: unusual amount + elevated recent activity + unusual destination behavior")


## 13. Common operational schema

The raw IEEE-CIS and PaySim schemas are different. The paper should therefore **not** claim direct feature equivalence.

Instead, use a common business layer:

| Business concept | IEEE-CIS | PaySim |
|---|---|---|
| Amount | `TransactionAmt` | `amount` |
| Time | `TransactionDT` | `step` |
| Transaction category | `ProductCD` | `type` |
| Customer/payment entity | `card1-card6` | `nameOrig` |
| Destination context | engineered/masked fields | `nameDest` |
| Fraud target | `isFraud` | `isFraud` |
| Behavioral history | C/D/V + derived | derived account history |

This supports a dataset-agnostic **risk → explanation → action** framework.

In [ ]:
harmonized = pd.DataFrame({
    "dataset": ["IEEE-CIS","PaySim"],
    "amount": ["TransactionAmt","amount"],
    "time": ["TransactionDT","step"],
    "transaction_type": ["ProductCD","type"],
    "entity": ["card1/card*","nameOrig"],
    "target": ["isFraud","isFraud"]
})
display(harmonized)


# 14. Preliminary feasibility verdict

The proposed paper is technically promising if the notebook confirms:

- [ ] IEEE-CIS files are found.
- [ ] PaySim is found at the supplied path.
- [ ] Both datasets contain `isFraud`.
- [ ] Temporal information is usable.
- [ ] Historical features can be generated without future leakage.
- [ ] The baseline has useful PR-AUC / recall / ranking performance.
- [ ] Risk scores can become `APPROVE / REVIEW / BLOCK`.
- [ ] Different business costs lead to meaningful policy differences.
- [ ] The same operational layer can work across both datasets.

### Target research pipeline

**Transaction → Behavioral Learning → Fraud Risk → Explanation → Business Action → Outcome/Feedback → Model Update**

If this notebook passes, the next stage should be a full experiment with IEEE-CIS as the primary dataset and PaySim as external/secondary validation.

## Dataset documentation

IEEE-CIS Fraud Detection: Kaggle dataset and official feature description. citeturn0search0turn0search1turn0search5

PaySim: Kaggle dataset metadata and documented headers. citeturn0search2